# Co-occurrence Preprocessing

Produce the two artefacts consumed downstream by the calibration loss
and by offline analysis, from `./{dataset}/handled/inter_seq.txt`:

- `co_occurrence.npz` — pairwise item co-occurrence on each user's
  **training** subsequence, log-normalized to `[0, 1]`, saved as a
  compressed NPZ (keys: `pairs` int32 `[N, 2]`, `weights` float32 `[N]`,
  plus `max_count` / `min_count` scalars). Loaded by
  `models.utils.FastCalibrationLoss` via `os.environ['CO_OCCURRENCE_FILE']`.
- `item_popularity.json` — per-item training frequency, kept as JSON for
  exploratory use only.

`inter_seq.txt` holds one line per user as
`user_id item_1 item_2 ... item_N`. The leave-two-out split in
`generators/generator.py` uses `item_N` for test and `item_{N-1}` for
validation, so we feed `tokens[1:-2]` (skip user_id + valid + test) into
both statistics to stay consistent with training.

Run this notebook from the `data/` directory so relative
`./<dataset>/handled/...` paths resolve.

In [ ]:
import json
import os
from collections import defaultdict

import numpy as np

DATASET = "fashion"
HANDLED_DIR = f"./{DATASET}/handled"
INPUT_FILE = os.path.join(HANDLED_DIR, "inter_seq.txt")
CO_OCC_FILE = os.path.join(HANDLED_DIR, "co_occurrence.npz")
POPULARITY_FILE = os.path.join(HANDLED_DIR, "item_popularity.json")

In [ ]:
def build_co_occurrence(input_file, output_file):
    """Compute pairwise item co-occurrence over each user's training
    subsequence (``tokens[1:-2]``, excluding user_id / valid / test),
    log-normalize to ``[0, 1]`` via ``log1p(count) / log1p(max_count)``,
    and save as a compressed NPZ with keys ``pairs``, ``weights``,
    ``max_count``, ``min_count``.
    """
    co_occurrence = defaultdict(int)
    min_item_id, max_item_id = None, None

    with open(input_file, "r") as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue
            items = [int(x) for x in tokens[1:-2]]
            if not items:
                continue

            lo, hi = min(items), max(items)
            min_item_id = lo if min_item_id is None else min(min_item_id, lo)
            max_item_id = hi if max_item_id is None else max(max_item_id, hi)

            for i in range(len(items)):
                for j in range(i + 1, len(items)):
                    a, b = sorted((items[i], items[j]))
                    co_occurrence[(a, b)] += 1

    if not co_occurrence:
        raise ValueError(f"No co-occurrence pairs found in {input_file}.")

    pairs = np.array(list(co_occurrence.keys()), dtype=np.int32)
    counts = np.array(list(co_occurrence.values()), dtype=np.int64)
    max_count, min_count = int(counts.max()), int(counts.min())
    weights = (np.log1p(counts) / np.log1p(max_count)).astype(np.float32)

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    np.savez_compressed(
        output_file,
        pairs=pairs,
        weights=weights,
        max_count=np.int64(max_count),
        min_count=np.int64(min_count),
    )

    print(f"Saved {len(pairs)} pairs to {output_file}")
    print(f"Item-id range: min={min_item_id}, max={max_item_id}")
    print(f"Pair-count range: min={min_count}, max={max_count}")


build_co_occurrence(INPUT_FILE, CO_OCC_FILE)

In [ ]:
def calculate_item_popularity(input_file, output_file):
    """Per-item training-portion frequency, saved as a JSON dict.

    Kept for offline analysis; not consumed by the training pipeline.
    """
    popularity = defaultdict(int)
    with open(input_file, "r") as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue
            for tok in tokens[1:-2]:
                popularity[int(tok)] += 1

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(dict(popularity), f, indent=2)

    print(f"Saved popularity for {len(popularity)} items to {output_file}")


calculate_item_popularity(INPUT_FILE, POPULARITY_FILE)